# Weather Features Gold Export

In [1]:
from pathlib import Path
import pandas as pd

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / "Data crawl").exists() else CWD.parent
if not (PROJECT_ROOT / "Data crawl").exists():
    raise FileNotFoundError("Cannot find project root containing 'Data crawl'. Open notebook from repo root or Source code/.")

SILVER_WEATHER_PATH = PROJECT_ROOT / "Data crawl" / "Silver_layer_2" / "Features" / "weather_features_hourly.csv"
GOLD_FEATURES_DIR = PROJECT_ROOT / "Data crawl" / "Gold_layer" / "Features"
GOLD_WEATHER_PATH = GOLD_FEATURES_DIR / "weather_features_hourly_gold.csv"
AUDIT_PATH = GOLD_FEATURES_DIR / "weather_features_gold_drop_audit.csv"

GOLD_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

DROP_COLUMNS = [
    "Is_Crosswind_15kt",
    "Is_Crosswind_20kt",
    "Is_Freezing",
    "Is_Gale_Wind",
    "Is_Strong_Wind",
    "Is_WMO_Fog_Code",
    "Is_WMO_Thunderstorm_Code",
    "Runway_Ice_Risk",
    "Is_Extreme_Heat",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SILVER_WEATHER_PATH:", SILVER_WEATHER_PATH)
print("GOLD_WEATHER_PATH:", GOLD_WEATHER_PATH)
print("DROP_COLUMNS:", DROP_COLUMNS)


PROJECT_ROOT: C:\Users\admin\Documents\DS108_AeroDelay
SILVER_WEATHER_PATH: C:\Users\admin\Documents\DS108_AeroDelay\Data crawl\Silver_layer_2\Features\weather_features_hourly.csv
GOLD_WEATHER_PATH: C:\Users\admin\Documents\DS108_AeroDelay\Data crawl\Gold_layer\Features\weather_features_hourly_gold.csv
DROP_COLUMNS: ['Is_Crosswind_15kt', 'Is_Crosswind_20kt', 'Is_Freezing', 'Is_Gale_Wind', 'Is_Strong_Wind', 'Is_WMO_Fog_Code', 'Is_WMO_Thunderstorm_Code', 'Runway_Ice_Risk', 'Is_Extreme_Heat']


## 1. Load Silver weather features

In [2]:
if not SILVER_WEATHER_PATH.exists():
    raise FileNotFoundError(f"Missing Silver weather features: {SILVER_WEATHER_PATH}")

weather = pd.read_csv(SILVER_WEATHER_PATH, low_memory=False)
print("Silver weather shape:", weather.shape)
display(weather.head())
display(pd.DataFrame({"column": weather.columns}))


Silver weather shape: (6624, 69)


                  time Airport  temperature  precipitation  cloudcover  wind_speed  wind_direction  pressure  humidity  visibility  dew_point_2m  weather_code  cape  lifted_index  cloud_cover_low  Visibility_SM  Visibility_Deficit_5KM_M  Visibility_Deficit_3SM_M  Visibility_Severity_Score  Dew_Point_Spread_C Wind_Sector  Wind_Runway_Relative_Angle_Deg  Crosswind_Kmh  Headwind_Kmh  Tailwind_Default_Runway_Kmh  Wind_Kt  Crosswind_Kt  Headwind_Kt  Tailwind_Default_Runway_Kt  Temp_Change_1H_C  Temp_Change_3H_C  Pressure_Change_3H_Hpa  Precip_Cumsum_1H_Mm  Precip_Cumsum_3H_Mm  Precip_Cumsum_6H_Mm  Wind_Gust_Estimate_Kmh  Crosswind_Max_3H_Kmh  Wind_Gust_Estimate_Kt  Crosswind_Max_3H_Kt  Gust_Variation_Kmh  Gust_Variation_Kt  Is_Rain  Is_Heavy_Rain  Is_Strong_Wind  Is_Gale_Wind  Is_Low_Visibility  Is_Fog  Is_Dewpoint_Spread_Le_1_5C  Is_Radiation_Fog_Risk  Is_WMO_Fog_Code  Is_WMO_Rain_Code  Is_WMO_Thunderstorm_Code  Is_Below_3SM_Visibility  Is_Below_1SM_Visibility  Is_Crosswind_10kt  Is_Crossw

                             column
0                              time
1                           Airport
2                       temperature
3                     precipitation
4                        cloudcover
5                        wind_speed
6                    wind_direction
7                          pressure
8                          humidity
9                        visibility
10                     dew_point_2m
11                     weather_code
12                             cape
13                     lifted_index
14                  cloud_cover_low
15                    Visibility_SM
16         Visibility_Deficit_5KM_M
17         Visibility_Deficit_3SM_M
18        Visibility_Severity_Score
19               Dew_Point_Spread_C
20                      Wind_Sector
21   Wind_Runway_Relative_Angle_Deg
22                    Crosswind_Kmh
23                     Headwind_Kmh
24      Tailwind_Default_Runway_Kmh
25                          Wind_Kt
26                     Cross

## 2. Drop selected columns and audit

In [3]:
drop_unique = list(dict.fromkeys(DROP_COLUMNS))
existing_drop_cols = [col for col in drop_unique if col in weather.columns]
missing_drop_cols = [col for col in drop_unique if col not in weather.columns]

weather_gold = weather.drop(columns=existing_drop_cols).copy()

audit = pd.DataFrame({
    "metric": [
        "silver_rows",
        "silver_columns",
        "gold_rows",
        "gold_columns",
        "dropped_columns_count",
        "missing_requested_drop_columns_count",
    ],
    "value": [
        len(weather),
        weather.shape[1],
        len(weather_gold),
        weather_gold.shape[1],
        len(existing_drop_cols),
        len(missing_drop_cols),
    ],
})

drop_detail = pd.DataFrame({
    "requested_drop_column": drop_unique,
    "was_present_in_silver": [col in existing_drop_cols for col in drop_unique],
})

print("Existing columns dropped:", existing_drop_cols)
print("Requested columns not found:", missing_drop_cols)
print("Gold weather shape:", weather_gold.shape)
display(audit)
display(drop_detail)


Existing columns dropped: ['Is_Crosswind_15kt', 'Is_Crosswind_20kt', 'Is_Freezing', 'Is_Gale_Wind', 'Is_Strong_Wind', 'Is_WMO_Fog_Code', 'Is_WMO_Thunderstorm_Code', 'Runway_Ice_Risk', 'Is_Extreme_Heat']
Requested columns not found: []
Gold weather shape: (6624, 60)


                                 metric  value
0                           silver_rows   6624
1                        silver_columns     69
2                             gold_rows   6624
3                          gold_columns     60
4                 dropped_columns_count      9
5  missing_requested_drop_columns_count      0

      requested_drop_column  was_present_in_silver
0         Is_Crosswind_15kt                   True
1         Is_Crosswind_20kt                   True
2               Is_Freezing                   True
3              Is_Gale_Wind                   True
4            Is_Strong_Wind                   True
5           Is_WMO_Fog_Code                   True
6  Is_WMO_Thunderstorm_Code                   True
7           Runway_Ice_Risk                   True
8           Is_Extreme_Heat                   True

## 3. Save Gold weather features

In [4]:
weather_gold.to_csv(GOLD_WEATHER_PATH, index=False, encoding="utf-8-sig")

# Save a compact audit table next to the output for reproducibility.
audit_out = pd.concat([
    audit.assign(section="summary"),
    drop_detail.rename(columns={"requested_drop_column": "metric", "was_present_in_silver": "value"}).assign(section="drop_detail"),
], ignore_index=True, sort=False)
audit_out.to_csv(AUDIT_PATH, index=False, encoding="utf-8-sig")

# Validate output headers.
written_cols = pd.read_csv(GOLD_WEATHER_PATH, nrows=0).columns.tolist()
remaining_drop_cols = [col for col in drop_unique if col in written_cols]
if remaining_drop_cols:
    raise AssertionError(f"Drop columns still present in Gold weather output: {remaining_drop_cols}")

print("Saved Gold weather features:", GOLD_WEATHER_PATH)
print("Saved drop audit:", AUDIT_PATH)
print("Remaining requested drop columns in output:", remaining_drop_cols)
print("Final shape:", (len(weather_gold), len(written_cols)))


Saved Gold weather features: C:\Users\admin\Documents\DS108_AeroDelay\Data crawl\Gold_layer\Features\weather_features_hourly_gold.csv
Saved drop audit: C:\Users\admin\Documents\DS108_AeroDelay\Data crawl\Gold_layer\Features\weather_features_gold_drop_audit.csv
Remaining requested drop columns in output: []
Final shape: (6624, 60)
